***Fase 1 - Preprocesamiento***

Objetivo
Preparar el dataset para el entrenamiento del MLP.


1. Importar librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score


2. Cargar dataset

In [ ]:

ruta = "../data/DataSet_completo.xlsx"

df = pd.read_excel(ruta)

print("- Head")
print(df.head())
print("\n- Shape")
print(df.shape)
print("\n- Info")
print(df.info())


3. Verificar valores nulos


In [ ]:
print(df.isnull().sum())

#Si fuera el caso
df = df.dropna()

4. Eliminar caractertíca dominante


In [ ]:
#El proyecto exige eliminar "Longitud"
df.drop(columns=["Longitud"], inplace = True)
print(df.info())

5. Separar variables


In [ ]:
X = df.drop(columns=["Clase"])
y = df["Clase"]

6. Estandarización


In [ ]:
#Usaremos Z-Score con StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

7. Visualización inicial

- Distribución de clases

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x=y)
plt.title("Distribución de clases")
plt.show()

- Matriz de correlación inicial

In [ ]:
corr = X.corr()

plt.figure(figsize=(16,12))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Matriz de correlación")
plt.show()

***Fase 2 - Clasificación inicial con MLP***

Objetivo: Entrenar el primer modelo MLP usando TODAS las características excepto "Longitud".


1. Crear modelo MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(100,50),
    activation='tanh',
    solver='sgd',
    max_iter=3800,
    random_state=42
)

2. Stratified K-Fold

In [ ]:
kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    mlp,
    X_scaled,
    y,
    cv=kfold,
    scoring='accuracy'
)

print("Accuracies:", scores)
print("Accuracy promedio:", scores.mean())

3. Metricas

In [ ]:
mlp.fit(X_scaled, y)

4. Matriz de confusión

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_test)


cm = confusion_matrix(y_test, y_pred, normalize='true')

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues')

plt.title("Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Valor real")

plt.show()

5. Curva de aprendizaje

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(mlp.loss_curve_)
plt.title("Curva de aprendizaje")
plt.xlabel("Iteraciones")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

6. Probar hiperparámetros

    Probar varias configuraciones

* Arquitecturas

* Activaciones

* Solvers